In [28]:
import json 
import pandas as pd
import re

In [29]:
with open(r'C:\Users\HP\OneDrive\Desktop\Bootcamp Materials\Projects\NLP\dataset.json','r') as f:
    data=json.load(f)

In [30]:
print(f"no.of chats:{len(data['customer_service_chats'])}")

no.of chats:15


In [31]:
data['customer_service_chats']

[{'chat_id': 'CHAT_001',
  'timestamp': '2024-11-15T10:23:45Z',
  'conversation': [{'speaker': 'Customer',
    'message': 'Hi, my name is Priya Sharma. I ordered a phone but received a tablet instead. Order number is ORD-445621.'},
   {'speaker': 'Executive',
    'message': 'Hello Priya, I apologize for the mix-up. May I have your contact details?'},
   {'speaker': 'Customer',
    'message': 'Sure, my phone is +91 97834 56721 and email is priya.sharma2024@gmail.com.'},
   {'speaker': 'Executive',
    'message': "Thank you. We'll arrange a replacement within 3 business days."}]},
 {'chat_id': 'CHAT_002',
  'timestamp': '2024-11-16T14:12:33Z',
  'conversation': [{'speaker': 'Customer',
    'message': 'My broadband connection keeps dropping every hour. Complaint ID: CMP2024-XY9988.'},
   {'speaker': 'Executive',
    'message': 'Sorry for the trouble. Could you provide your customer ID and phone number?'},
   {'speaker': 'Customer',
    'message': 'Customer ID is CUST-88991, phone is 080-4

In [130]:
patterns = {
    "email": r"([A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.(?:com|in|org|net|edu|gov|co\.in))(?=\s|$|[.,])",

    "phone": r"(?:\+91[-\s]?)?\d{4,5}[-\s]\d{2,3}[-\s]\d{3,4}"
             r"|(?:\+1[-\s]?)?\(?\d{3}\)?[-\s]?\d{3}[-\s]?\d{4}"
             r"|\b\d{10}\b",

    "order_id": r"\b(?:Order|ORD)[-#\s]*[A-Z0-9]{5,}[-]?[A-Z]?\b",
    "ticket_id": r"\bTKT#?\d{4}[-]?\d{4}\b",
    "complaint_id": r"\bCMP\d{4}[-]?[A-Z]{2}\d{4}\b",
    "customer_id": r"\bCUST[-]?\d{5,6}\b",
    "transaction_id": r"\bTXN[-]?\d{6}[-]?[A-Z]{2}\b",
    "service_request": r"\bSR[-]?\d{4}[-]?[A-Z]{2}\b",
    "subscription_id": r"\bSUB[-]?\d{6}[-]?[A-Z]\b",
    "booking_ref": r"\bBK[-]?\d{5}[-]?[A-Z]\b",
    "policy_number": r"\bPOL[-]?\d{6}\b",
    "claim_number": r"\bCLM[-]?\d{4}[-]?\d{6}\b",
    "reference_number": r"\bREF[-]?\d{6}[-]?[A-Z]\b",
    "return_id": r"\bRTN[-]?\d{6}[-]?[A-Z]\b",
    "cancellation_id": r"\bCANC[-]?\d{4}[-]?[A-Z]\b",

    # PREFIX REQUIRED → prevents "switching providers"
      # ---------- NAME (PREFIX REQUIRED) ----------
    "name_with_prefix": (
        r"(?:\bmy name is\b|\bi am\b|\bi'm\b|\bname is\b|\bname:\b|\baccount holder:\b)\s+"
        r"([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)"
    ),

    # ---------- HIGH-CONFIDENCE FALLBACK ----------
    "name_before_contact": (
        r"\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\b(?=,\s*(?:phone|email))"
    )
}


In [131]:
import re

def extract_pii(conversation):
    full_text = " ".join(msg["message"] for msg in conversation)

    pii = {
        "emails": [],
        "phone": [],
        "names": []
    }

    # ---------- EMAIL & PHONE ----------
    pii["emails"] = list(set(re.findall(patterns["email"], full_text, re.IGNORECASE)))
    pii["phone"] = list(set(re.findall(patterns["phone"], full_text)))

    # ---------- NAME EXTRACTION ----------
    name_set = set()

    for msg in conversation:
        # Only extract names from customer messages
        if msg.get("speaker", "").lower() != "customer":
            continue

        text = msg["message"]

        # Strong pattern (prefix REQUIRED)
        prefixed_names = re.findall(
            patterns["name_with_prefix"],
            text,
            flags=re.IGNORECASE
        )

        # High-confidence fallback only
        fallback_names = re.findall(
            patterns["name_before_contact"],
            text
        )

        for name in prefixed_names + fallback_names:
            clean_name = name.strip()
            if clean_name:
                name_set.add(clean_name)

    pii["names"] = list(name_set)

    return pii


In [132]:
def extract_ids(conversation):
    full_text=' '.join([msg['message'] for msg in conversation])
    ids={}
    id_types =['order_id', 'ticket_id', 'complaint_id', 'customer_id', 
                'transaction_id', 'service_request', 'subscription_id',
                'booking_ref', 'policy_number', 'claim_number', 
                'reference_number', 'return_id']
    for id_type in id_types:
        matches=re.findall(patterns[id_type],full_text,re.IGNORECASE)
        if matches:
            ids[id_type]= list(set(matches))
    return ids

In [133]:
def extract_customer_query(conversation):
    customer_messages = [msg['message']for msg in conversation if msg['speaker']=='Customer']
    if customer_messages:
        return customer_messages[0]
    return ""


In [134]:
def extract_resolution(converstaion):
    executive_messages=[msg['message']for msg in converstaion if msg['speaker']== 'Executive']
    if executive_messages:
        #-1 because the last message has always the resolution
        return executive_messages[-1] 
    return ""


In [135]:
def categorize_query(query):
    query_lower=query.lower()
    categories = {
        'Order Issue': ['order', 'delivery', 'shipment', 'package', 'received'],
        'Refund': ['refund', 'money back', 'return'],
        'Technical Support': ['not working', 'error', 'can\'t access', 'locked out', 'login'],
        'Billing': ['charged', 'payment', 'transaction', 'billing'],
        'Cancellation': ['cancel', 'cancellation', 'unsubscribe'],
        'Account Access': ['account', 'password', 'locked', 'access'],
        'Service Request': ['technician', 'repair', 'service', 'fix'],
        'Complaint': ['complaint', 'issue', 'problem'],
        'Information': ['update', 'change', 'modify', 'information']
    }

    for category, keywords in categories.items():
        if any(keyword in query_lower for keyword in keywords):
            return category
        return 'other'

In [136]:
processed_data = []
for chat in data['customer_service_chats']:
    chat_id = chat['chat_id']
    timestamp = chat['timestamp']
    conversation = chat['conversation']
    
    # Extract all information
    pii = extract_pii(conversation)
    ids = extract_ids(conversation)
    query = extract_customer_query(conversation)
    resolution = extract_resolution(conversation)
    category = categorize_query(query)

    # Create a record
    record = {
        'chat_id': chat_id,
        'timestamp': timestamp,
        'category': category,
        'customer_query': query,
        'resolution': resolution,
        'customer_name': ', '.join(pii['names']) if pii['names'] else '',
        'email': ', '.join(pii['emails']) if pii['emails'] else '',
        'phone': ', '.join(pii['phone']) if pii['phone'] else '',
        'order_id': ', '.join(ids.get('order_id', [])),
        'ticket_id': ', '.join(ids.get('ticket_id', [])),
        'complaint_id': ', '.join(ids.get('complaint_id', [])),
        'customer_id': ', '.join(ids.get('customer_id', [])),
        'transaction_id': ', '.join(ids.get('transaction_id', [])),
        'service_request': ', '.join(ids.get('service_request', [])),
        'subscription_id': ', '.join(ids.get('subscription_id', [])),
        'booking_ref': ', '.join(ids.get('booking_ref', [])),
        'policy_number': ', '.join(ids.get('policy_number', [])),
        'claim_number': ', '.join(ids.get('claim_number', [])),
        'reference_number': ', '.join(ids.get('reference_number', [])),
        'return_id': ', '.join(ids.get('return_id', []))
    }
    
    processed_data.append(record)

In [137]:
df = pd.DataFrame(processed_data)

print(f"Total chats processed: {len(df)}")
print(df['chat_id'].tolist())

Total chats processed: 15
['CHAT_001', 'CHAT_002', 'CHAT_003', 'CHAT_004', 'CHAT_005', 'CHAT_006', 'CHAT_007', 'CHAT_008', 'CHAT_009', 'CHAT_010', 'CHAT_011', 'CHAT_012', 'CHAT_013', 'CHAT_014', 'CHAT_015']


In [138]:
df

,chat_id,timestamp,category,customer_query,resolution,customer_name,email,phone,order_id,ticket_id,complaint_id,customer_id,transaction_id,service_request,subscription_id,booking_ref,policy_number,claim_number,reference_number,return_id
0,CHAT_001,2024-11-15T10:23:45Z,Order Issue,"Hi, my name is Priya Sharma. I ordered a phone...",Thank you. We'll arrange a replacement within ...,Priya Sharma,priya.sharma2024@gmail.com,,"ORD-445621, Order number",,,,,,,,,,,
1,CHAT_002,2024-11-16T14:12:33Z,other,My broadband connection keeps dropping every h...,And your registered email address?,,vijay_kumar@hotmail.com,9876-543-210,,,CMP2024-XY9988,CUST-88991,,,,,,,,
2,CHAT_003,2024-11-17T09:45:12Z,other,"Hello, I need to cancel my subscription. My ac...",Got it. May I know the reason for cancellation?,switching providers,sarah.johnson@live.com,+1-415-555-0188,,,,,,,,,,,,
3,CHAT_004,2024-11-18T16:30:27Z,other,My AC isn't cooling properly. Service request ...,We'll schedule a technician visit within 24 ho...,Arjun Menon,arjun.menon1985@yahoo.in,"080 2233445, +91-9123-456-789",,,,,,SR-3344-AB,,,,,,
4,CHAT_005,2024-11-19T11:05:56Z,other,I was charged twice for the same transaction. ...,Do you have a ticket number for this issue?,,jessica_brown@protonmail.com,(212) 555-0145,,TKT#8877-3322,,,TXN-998877-CC,,,,,,,
5,CHAT_006,2024-11-20T13:22:40Z,Order Issue,My delivery was supposed to arrive yesterday b...,Let me check. Your package is out for delivery...,Amit Patel,amit.patel.delhi@gmail.com,+91-9988-77-6655,ORD-334455-B,,,,,,,,,,,
6,CHAT_007,2024-11-21T08:15:19Z,other,I can't access my account after the recent upd...,And your phone number?,,linda_garcia@outlook.com,(310) 555-0167,,,,,,,,,,,,
7,CHAT_008,2024-11-22T15:40:52Z,Order Issue,The product I received is defective. I want a ...,Got it. Do you have any photos of the defect?,Sneha Reddy,sneha.r2024@yahoo.co.in,"+91-8877-665-544, 9100-223-344",,,,,,,,,,,,
8,CHAT_009,2024-11-23T10:55:31Z,other,I need technical support for my printer. It's ...,Thank you. Have you tried restarting the device?,Robert Chen,robert.chen@fastmail.com,(650) 555-0192,,TKT#9988-7766,,,,,,,,,,
9,CHAT_010,2024-11-24T12:18:44Z,other,My flight booking got cancelled but I didn't r...,Was this booking done through our app or website?,,maria.lopez@icloud.com,+1-305-555-0123,,,,CUST-445566,,,,BK-77889-Q,,,,


In [139]:
import pandas as pd

# Load your data (assuming it's in df)
# Display settings for better viewing
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', None)

# View specific columns clearly
print("="*80)
print("PII EXTRACTION RESULTS")
print("="*80)

# Check PII extraction quality
pii_cols = ['chat_id', 'customer_name', 'email', 'phone']
print("\n1. PII Data:")
print(df[pii_cols].head(10))

# Check ID extraction
id_cols = ['chat_id', 'order_id', 'ticket_id', 'subscription_id', 'customer_id']
print("\n2. ID Extraction:")
print(df[id_cols].head(10))

# Check categorization
print("\n3. Category Distribution:")
print(df['category'].value_counts())

# Check for missing data
print("\n4. Data Completeness:")
print(f"Total chats: {len(df)}")
print(f"Chats with emails: {df['email'].notna().sum()}")
print(f"Chats with phones: {df['phone'].notna().sum()}")
print(f"Chats with names: {df['customer_name'].notna().sum()}")

# View full details of a specific chat
print("\n5. Sample Chat Detail (CHAT_001):")
chat_detail = df[df['chat_id'] == 'CHAT_001'].T
print(chat_detail)

# Export clean view
df.to_excel('chat_analysis_clean.xlsx', index=False)
print("\n✓ Clean export saved to 'chat_analysis_clean.xlsx'")

PII EXTRACTION RESULTS

1. PII Data:
    chat_id        customer_name                         email  \
0  CHAT_001         Priya Sharma    priya.sharma2024@gmail.com   
1  CHAT_002                            vijay_kumar@hotmail.com   
2  CHAT_003  switching providers        sarah.johnson@live.com   
3  CHAT_004          Arjun Menon      arjun.menon1985@yahoo.in   
4  CHAT_005                       jessica_brown@protonmail.com   
5  CHAT_006           Amit Patel    amit.patel.delhi@gmail.com   
6  CHAT_007                           linda_garcia@outlook.com   
7  CHAT_008          Sneha Reddy       sneha.r2024@yahoo.co.in   
8  CHAT_009          Robert Chen      robert.chen@fastmail.com   
9  CHAT_010                             maria.lopez@icloud.com   

                            phone  
0                                  
1                    9876-543-210  
2                 +1-415-555-0188  
3   080 2233445, +91-9123-456-789  
4                  (212) 555-0145  
5                +91